# Training Pipeline API — Testes

Testa todos os endpoints de `/pipeline`:
- `POST /pipeline/segmentation` — Iniciar segmentação
- `POST /pipeline/profiles` — Gerar perfis
- `POST /pipeline/datasets` — Gerar datasets
- `POST /pipeline/optimization` — Otimização + treino
- `POST /pipeline/full` — Pipeline completo
- `GET /pipeline/runs/{run_id}` — Consultar execução
- `GET /pipeline/runs` — Listar execuções

In [50]:
import httpx
import time
from pathlib import Path

BASE_URL = "http://localhost:8010"
client = httpx.Client(base_url=BASE_URL, timeout=600)

In [51]:
TARGET = "h"
DATA_PATH = Path('../../../datasets/full/telemetria_movias2025_features.csv')

## 1. Segmentação

In [52]:

resp = client.post("/pipeline/segmentation", json={"target": TARGET, "data_path": str(DATA_PATH.resolve())})
print(f"Status: {resp.status_code}")
seg_run = resp.json()
seg_run

Status: 202


{'id': 14,
 'step': 'segmentation',
 'target': 'h',
 'model_type': None,
 'status': 'pending',
 'started_at': None,
 'finished_at': None,
 'error_message': None,
 'metrics': None,
 'artifacts': None,
 'created_at': '2026-04-26T23:14:35'}

In [ ]:
# Consultar estado da execução
seg_run = resp.json()
run_id = seg_run['id']
resp = client.get(f"/pipeline/runs/{run_id}")
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'id': 14,
 'step': 'segmentation',
 'target': 'h',
 'model_type': None,
 'status': 'completed',
 'started_at': '2026-04-26T23:14:35.860779',
 'finished_at': '2026-04-26T23:15:34.195680',
 'error_message': None,
 'metrics': None,
 'artifacts': {'segmentation_dir': 'C:\\Users\\f0pi\\git\\apimovias\\logs\\segmentation',
  'classification_dir': 'C:\\Users\\f0pi\\git\\apimovias\\models\\classification',
  'train_data_dir': 'C:\\Users\\f0pi\\git\\apimovias\\data\\train_data'},
 'created_at': '2026-04-26T23:14:35'}

## 2. Perfis

In [41]:
resp = client.post("/pipeline/profiles", json={"target": TARGET})
print(f"Status: {resp.status_code}")
prof_run = resp.json()
prof_run

Status: 202


{'id': 11,
 'step': 'profiles',
 'target': 'h',
 'model_type': None,
 'status': 'pending',
 'started_at': None,
 'finished_at': None,
 'error_message': None,
 'metrics': None,
 'artifacts': None,
 'created_at': '2026-04-26T22:14:41'}

In [42]:
run_id = prof_run["id"]
resp = client.get(f"/pipeline/runs/{run_id}")
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'id': 11,
 'step': 'profiles',
 'target': 'h',
 'model_type': None,
 'status': 'completed',
 'started_at': '2026-04-26T22:14:41.408493',
 'finished_at': '2026-04-26T22:15:08.801240',
 'error_message': None,
 'metrics': None,
 'artifacts': {'profile_dir': 'C:\\Users\\f0pi\\git\\apimovias\\data\\profiles\\h'},
 'created_at': '2026-04-26T22:14:41'}

## 3. Datasets

In [43]:
resp = client.post("/pipeline/datasets", json={"target": TARGET})
print(f"Status: {resp.status_code}")
ds_run = resp.json()
ds_run

Status: 202


{'id': 12,
 'step': 'dataset',
 'target': 'h',
 'model_type': None,
 'status': 'pending',
 'started_at': None,
 'finished_at': None,
 'error_message': None,
 'metrics': None,
 'artifacts': None,
 'created_at': '2026-04-26T22:15:55'}

In [44]:
run_id = ds_run["id"]
resp = client.get(f"/pipeline/runs/{run_id}")
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'id': 12,
 'step': 'dataset',
 'target': 'h',
 'model_type': None,
 'status': 'completed',
 'started_at': '2026-04-26T22:15:55.048123',
 'finished_at': '2026-04-26T22:16:03.245714',
 'error_message': None,
 'metrics': None,
 'artifacts': {'cache_dir': 'C:\\Users\\f0pi\\git\\apimovias\\data\\.dataset_cache\\h',
  'n_samples': 11163,
  'n_general_features': 77,
  'n_recent_features': 28},
 'created_at': '2026-04-26T22:15:55'}

## 4. Otimização

Requer `model_type`: `"multihead"` ou `"moe"`.

In [45]:
MODEL_TYPE = "multihead"

resp = client.post("/pipeline/optimization", json={"target": TARGET, "model_type": MODEL_TYPE})
print(f"Status: {resp.status_code}")
opt_run = resp.json()
opt_run

Status: 202


{'id': 13,
 'step': 'optimization',
 'target': 'h',
 'model_type': 'multihead',
 'status': 'pending',
 'started_at': None,
 'finished_at': None,
 'error_message': None,
 'metrics': None,
 'artifacts': None,
 'created_at': '2026-04-26T22:17:28'}

In [46]:
run_id = opt_run["id"]
resp = client.get(f"/pipeline/runs/{run_id}")
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'id': 13,
 'step': 'optimization',
 'target': 'h',
 'model_type': 'multihead',
 'status': 'running',
 'started_at': '2026-04-26T22:17:28.889692',
 'finished_at': None,
 'error_message': None,
 'metrics': None,
 'artifacts': None,
 'created_at': '2026-04-26T22:17:28'}

## 5. Pipeline completo

Executa segmentação → perfis → datasets → otimização em sequência.

In [ ]:
resp = client.post("/pipeline/full", json={
    "target": TARGET,
    "data_path": DATA_PATH,
    "model_type": MODEL_TYPE,
})
print(f"Status: {resp.status_code}")
full_runs = resp.json()
full_runs

## 6. Listar execuções

In [57]:
import pandas as pd

# Todas as execuções
resp = client.get("/pipeline/runs")
print(f"Status: {resp.status_code}")
df_runs = pd.DataFrame(resp.json())
display(df_runs)

Status: 200


,id,step,target,model_type,status,started_at,finished_at,error_message,metrics,artifacts,created_at
0,4,ingestion,global,None,completed,2026-04-27T00:39:49.883467,2026-04-27T00:42:20.958032,None,{'ingestion': {'daily_activity_inserted': 4621...,None,2026-04-27T00:39:49
1,3,ingestion,global,None,completed,2026-04-27T00:37:10.616027,2026-04-27T00:39:38.874832,None,{'ingestion': {'daily_activity_inserted': 4580...,None,2026-04-27T00:37:10
2,2,ingestion,global,None,completed,2026-04-27T00:34:18.431674,2026-04-27T00:36:50.308909,None,{'ingestion': {'daily_activity_inserted': 4612...,None,2026-04-27T00:34:18
3,1,ingestion,global,None,completed,2026-04-27T00:31:12.747795,2026-04-27T00:34:01.265706,None,{'ingestion': {'daily_activity_inserted': 1303...,None,2026-04-27T00:31:12


In [ ]:
# Filtrar por step e target
resp = client.get("/pipeline/runs", params={"step": "segmentation", "target": TARGET})
print(f"Status: {resp.status_code}")
pd.DataFrame(resp.json())

In [ ]:
# Filtrar por status
resp = client.get("/pipeline/runs", params={"status": "completed"})
print(f"Status: {resp.status_code}")
pd.DataFrame(resp.json())

## 7. Consultar execução por ID

In [ ]:
RUN_ID = 1

resp = client.get(f"/pipeline/runs/{RUN_ID}")
print(f"Status: {resp.status_code}")
resp.json()

In [ ]:
# Run inexistente → 404
resp = client.get("/pipeline/runs/999999")
print(f"Status: {resp.status_code}")
resp.json()

## 8. Testar conflito (409)

Submeter a mesma etapa enquanto ainda está a correr.

In [ ]:
# Primeiro, submeter segmentação
resp1 = client.post("/pipeline/segmentation", json={"target": TARGET, "data_path": DATA_PATH})
print(f"Primeira submissão: {resp1.status_code}")

# Imediatamente, submeter outra vez
resp2 = client.post("/pipeline/segmentation", json={"target": TARGET, "data_path": DATA_PATH})
print(f"Segunda submissão: {resp2.status_code}  (esperado: 409 se ainda a correr)")
resp2.json()